# 第五课：NumPy 从已有数据创建数组

本课依据菜鸟教程的 [NumPy 从已有的数组创建数组](https://www.runoob.com/numpy/numpy-array-from-existing-data.html) 整理，学习 `array()`、`asarray()`、`frombuffer()` 和 `fromiter()`，并重点理解复制与共享内存。

In [1]:
import numpy as np
print("NumPy 版本：", np.__version__)

NumPy 版本： 2.5.3


## 1. 从列表或元组创建数组

`np.array()` 和 `np.asarray()` 都可以把 Python 列表、元组等序列转换成 `ndarray`。

In [2]:
list_data = [1, 2, 3]
tuple_data = (4, 5, 6)
matrix_data = [[1, 2], [3, 4]]
print(np.asarray(list_data))
print(np.asarray(tuple_data, dtype=np.float64))
print(np.asarray(matrix_data))

[1 2 3]
[4. 5. 6.]
[[1 2]
 [3 4]]


多维数值数组要求嵌套序列形状规则一致。像 `[[1, 2, 3], [4, 5]]` 这样的参差结构不能直接成为普通二维数值数组；NumPy 2.x 通常会报错。

In [3]:
ragged = [[1, 2, 3], [4, 5]]
object_array = np.array(ragged, dtype=object)
print(object_array)
print("dtype：", object_array.dtype)
print("shape：", object_array.shape)

[list([1, 2, 3]) list([4, 5])]
dtype： object
shape： (2,)


## 2. `array()` 与 `asarray()` 的关键区别

输入已经是合适的 `ndarray` 时：

- `np.asarray(x)` 通常直接返回原数组，不复制数据。
- `np.array(x)` 默认创建副本。

因此，修改 `asarray()` 的结果可能同时修改原数组。不要仅用 `is` 判断复杂场景，`np.shares_memory()` 更能表达是否共享底层数据。

In [4]:
original = np.array([10, 20, 30])
by_array = np.array(original)
by_asarray = np.asarray(original)
print("array 共享内存：", np.shares_memory(original, by_array))
print("asarray 共享内存：", np.shares_memory(original, by_asarray))
by_asarray[0] = 999
print("修改 asarray 结果后的原数组：", original)

array 共享内存： False
asarray 共享内存： True
修改 asarray 结果后的原数组： [999  20  30]


需要明确获得独立副本时，可以使用 `x.copy()` 或 `np.array(x, copy=True)`。需要尽量避免复制时，使用 `np.asarray(x)`。如果转换 `dtype`，即使使用 `asarray()` 也往往必须创建新数据。

In [4]:
source = np.array([1, 2, 3], dtype=np.int32)
same_type = np.asarray(source, dtype=np.int32)
new_type = np.asarray(source, dtype=np.float64)
print("同类型共享：", np.shares_memory(source, same_type))
print("转换类型共享：", np.shares_memory(source, new_type))

同类型共享： True
转换类型共享： False


## 3. `frombuffer()`：解释二进制缓冲区

`np.frombuffer(buffer, dtype, count=-1, offset=0)` 不逐个解析文本，而是按指定 `dtype` 直接解释底层字节。`dtype` 必须与数据的实际编码一致。

In [9]:
message = b"Hello"
letters = np.frombuffer(message, dtype="S1")
codes = np.frombuffer(message, dtype=np.uint8)
print(letters)
print(codes)

[b'H' b'e' b'l' b'l' b'o']
[ 72 101 108 108 111]


In [5]:
raw = bytes([10, 20, 30, 40, 50, 60])
print("全部读取：", np.frombuffer(raw, dtype=np.uint8))
print("跳过前 2 字节，读取 3 个：", np.frombuffer(raw, dtype=np.uint8, count=3, offset=2))

全部读取： [10 20 30 40 50 60]
跳过前 2 字节，读取 3 个： [30 40 50]


`frombuffer()` 创建的是原始缓冲区的视图。不可变 `bytes` 产生的数组通常只读；可变 `bytearray` 产生的数组可以修改，并会影响原缓冲区。

In [ ]:
mutable_buffer = bytearray([1, 2, 3, 4])#多了一个合格相当于使用frombuffer后有单独的内存空间
buffer_view = np.frombuffer(mutable_buffer, dtype=np.uint8)
buffer_view[0] = 99
print("数组：", buffer_view)
print("原 bytearray：", list(mutable_buffer))

数组： [99  2  3  4]
原 bytearray： [99, 2, 3, 4]


## 4. `fromiter()`：从可迭代对象创建一维数组

`np.fromiter(iterable, dtype, count=-1)` 适用于生成器和迭代器。必须指定 `dtype`，返回值始终是一维数组。已知元素数量时提供 `count` 可以减少内存重新分配。

In [12]:
squares = np.fromiter((x * x for x in range(6)), dtype=np.int64)
first_three = np.fromiter((x / 10 for x in range(100)), dtype=np.float64, count=3)
print(squares)
print(first_three)

[ 0  1  4  9 16 25]
[0.  0.1 0.2]


## 如何选择？

| 输入来源或目的 | 推荐方式 |
|---|---|
| Python 列表、元组 | `np.array()` / `np.asarray()` |
| 想要明确副本 | `np.array(..., copy=True)` / `.copy()` |
| 已是数组，想尽量避免复制 | `np.asarray()` |
| 原始二进制缓冲区 | `np.frombuffer()` |
| 生成器或迭代器 | `np.fromiter()` |

## 本节练习

1. 用 `asarray()` 将列表 `[[1, 2], [3, 4]]` 转为 `float32`。
2. 为它创建独立副本，修改副本后验证原数组不变。
3. 使用 `fromiter()` 创建前 8 个偶数。
4. 使用 `frombuffer()` 把 `b"NumPy"` 拆成单字节字符串数组。

In [18]:
data = [[1, 2], [3, 4]]
# 请在这里完成练习
data_as = np.asarray(data, dtype=np.float32)
data_ar = np.array(data, copy=True)
print(data)
data_ar[0, 0] = 99
print( data)
data_ou = np.fromiter((i for i in range(20) if i % 2 == 0), dtype=np.int16,count=8)
print(data_ou)
data = np.frombuffer(b"Numpy" ,dtype="S1")
print(data)


[[1, 2], [3, 4]]
[[1, 2], [3, 4]]
[ 0  2  4  6  8 10 12 14]
[b'N' b'u' b'm' b'p' b'y']


## 参考答案

In [16]:
data = [[1, 2], [3, 4]]
arr = np.asarray(data, dtype=np.float32)
independent = arr.copy()
independent[0, 0] = 100
evens = np.fromiter((2 * x for x in range(8)), dtype=np.int32)
word = np.frombuffer(b"NumPy", dtype="S1")
print("原数组：\n", arr)
print("修改后的副本：\n", independent)
print("共享内存：", np.shares_memory(arr, independent))
print("偶数：", evens)
print("字节字符：", word)

原数组：
 [[1. 2.]
 [3. 4.]]
修改后的副本：
 [[100.   2.]
 [  3.   4.]]
共享内存： False
偶数： [ 0  2  4  6  8 10 12 14]
字节字符： [b'N' b'u' b'm' b'P' b'y']


---

下一课：[NumPy 从数值范围创建数组](https://www.runoob.com/numpy/numpy-array-from-numerical-ranges.html)。